## Imports

In [ ]:
pip install torch transformers

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

from __future__ import annotations
from dataclasses import dataclass
from abc import ABC, abstractmethod

from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

## Wrapper for GPT-2 model

In [ ]:
class GPT2DiffusionTransformer(nn.Module):
    """
    GPT-2 przerobiony na bidirectional denoising Transformer.

    Wejście:
        corrupted_ids:  [batch, sequence]
        timesteps:      [batch]
        attention_mask: [batch, sequence]

    Wyjście:
        logits: [batch, sequence, vocabulary]

    Model przewiduje czysty token na tej samej pozycji:
        logits[:, i, :] -> clean_ids[:, i]
    """

    def __init__(
        self,
        model_name: str = "openai-community/gpt2",
        num_diffusion_steps: int = 1000,
        vocabulary_size: int | None = None,
    ) -> None:
        super().__init__()

        pretrained = AutoModelForCausalLM.from_pretrained(
            model_name,
            attn_implementation="eager",
        )

        if vocabulary_size is not None:
            pretrained.resize_token_embeddings(
                vocabulary_size
            )

        self.transformer = pretrained.transformer
        self.lm_head = pretrained.lm_head
        self.config = pretrained.config

        self.time_embedding = nn.Embedding(
            num_diffusion_steps,
            self.config.n_embd,
        )

        self.num_diffusion_steps = num_diffusion_steps

    def forward(
        self,
        corrupted_ids: torch.Tensor,
        timesteps: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        self._validate_inputs(
            corrupted_ids=corrupted_ids,
            timesteps=timesteps,
            attention_mask=attention_mask,
        )

        batch_size, sequence_length = corrupted_ids.shape

        # Handle the case where sequence_length is 0
        if sequence_length == 0:
            # Return an empty tensor of the expected shape for logits
            return torch.empty(
                batch_size,
                0,
                self.config.vocab_size,
                device=corrupted_ids.device,
                dtype=self.lm_head.weight.dtype
            )

        token_embeddings = self.transformer.wte(corrupted_ids)
        time_embeddings = self.time_embedding(timesteps).unsqueeze(1)
        input_embeddings = token_embeddings + time_embeddings

        bidirectional_mask = torch.zeros(
            batch_size,
            1,
            sequence_length,
            sequence_length,
            device=input_embeddings.device,
            dtype=input_embeddings.dtype,
        )

        if attention_mask is not None:
            padding_mask = attention_mask[:, None, None, :].bool()

            bidirectional_mask = (
                bidirectional_mask.masked_fill(
                    ~padding_mask,
                    torch.finfo(
                        input_embeddings.dtype
                    ).min,
                )
            )

        outputs = self.transformer(
            inputs_embeds=input_embeddings,
            attention_mask=bidirectional_mask,
            use_cache=False,
            return_dict=True,
        )

        hidden_states = outputs.last_hidden_state
        logits = self.lm_head(hidden_states)
        return logits

    @torch.no_grad()
    def denoise(
        self,
        corrupted_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        corrupted_positions: torch.Tensor | None = None,
        num_iterations: int = 10,
        temperature: float = 1.0,
    ) -> torch.Tensor:
        """
        Iteracyjnie poprawia zaburzoną sekwencję.

        Jeżeli corrupted_positions jest podane,
        aktualizowane są tylko wskazane pozycje.

        Jeżeli corrupted_positions jest None,
        aktualizowane są wszystkie niepaddingowe pozycje.
        """
        if corrupted_ids.ndim != 2:
            raise ValueError(
                "corrupted_ids must have shape "
                "[batch, sequence]"
            )

        if num_iterations <= 0:
            raise ValueError(
                "num_iterations must be greater than zero"
            )

        if temperature <= 0:
            raise ValueError(
                "temperature must be greater than zero"
            )

        device = next(self.parameters()).device

        reconstructed_ids = corrupted_ids.clone().to(
            device
        )

        if attention_mask is None:
            attention_mask = torch.ones_like(
                reconstructed_ids,
                dtype=torch.long,
                device=device,
            )
        else:
            attention_mask = attention_mask.to(
                device
            )

        if corrupted_positions is None:
            positions_to_update = attention_mask.bool()
        else:
            if (
                corrupted_positions.shape
                != reconstructed_ids.shape
            ):
                raise ValueError(
                    "corrupted_positions must have the "
                    "same shape as corrupted_ids"
                )

            positions_to_update = (
                corrupted_positions.to(device).bool()
                & attention_mask.bool()
            )

        if not positions_to_update.any():
            return reconstructed_ids

        was_training = self.training
        self.eval()

        active_positions = positions_to_update.clone()

        for iteration in range(num_iterations):
            if not active_positions.any():
                break

            timestep = round(
                (
                    self.num_diffusion_steps - 1
                )
                * (
                    1.0
                    -
                    iteration
                    /
                    max(
                        num_iterations - 1,
                        1,
                    )
                )
            )

            timesteps = torch.full(
                size=(reconstructed_ids.size(0),),
                fill_value=timestep,
                dtype=torch.long,
                device=device,
            )

            logits = self.forward(
                corrupted_ids=reconstructed_ids,
                timesteps=timesteps,
                attention_mask=attention_mask,
            )

            logits = logits / temperature

            probabilities = torch.softmax(
                logits,
                dim=-1,
            )

            confidence, predicted_ids = (
                probabilities.max(dim=-1)
            )

            for batch_idx in range(
                reconstructed_ids.size(0)
            ):
                current_positions = active_positions[
                    batch_idx
                ].nonzero(as_tuple=True)[0]

                if current_positions.numel() == 0:
                    continue

                remaining_iterations = (
                    num_iterations - iteration
                )

                number_to_update = max(
                    1,
                    int(
                        current_positions.numel()
                        /
                        remaining_iterations
                    ),
                )

                current_confidence = confidence[
                    batch_idx,
                    current_positions,
                ]

                selected_local_indices = torch.topk(
                    current_confidence,
                    k=min(
                        number_to_update,
                        current_positions.numel(),
                    ),
                ).indices

                selected_positions = current_positions[
                    selected_local_indices
                ]

                reconstructed_ids[
                    batch_idx,
                    selected_positions,
                ] = predicted_ids[
                    batch_idx,
                    selected_positions,
                ]

                active_positions[
                    batch_idx,
                    selected_positions,
                ] = False

        if was_training:
            self.train()

        return reconstructed_ids

    @staticmethod
    def compute_loss(
        logits: torch.Tensor,
        clean_ids: torch.Tensor,
        corrupted_positions: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Rekonstrukcja tokenu na tej samej pozycji.

        logits[:, i, :] porównujemy z clean_ids[:, i].
        """
        if logits.ndim != 3:
            raise ValueError(
                "logits must have shape "
                "[batch, sequence, vocabulary]"
            )

        if clean_ids.ndim != 2:
            raise ValueError(
                "clean_ids must have shape "
                "[batch, sequence]"
            )

        if logits.shape[:2] != clean_ids.shape:
            raise ValueError(
                "The first two dimensions of logits "
                "must match clean_ids"
            )

        labels = clean_ids.clone()

        if corrupted_positions is not None:
            if corrupted_positions.shape != clean_ids.shape:
                raise ValueError(
                    "corrupted_positions must have the "
                    "same shape as clean_ids"
                )

            labels[
                ~corrupted_positions.bool()
            ] = -100

        if attention_mask is not None:
            if attention_mask.shape != clean_ids.shape:
                raise ValueError(
                    "attention_mask must have the same "
                    "shape as clean_ids"
                )

            labels[
                attention_mask == 0
            ] = -100

        valid_positions = labels != -100

        if not valid_positions.any():
            return logits.sum() * 0.0

        return F.cross_entropy(
            logits.reshape(
                -1,
                logits.size(-1),
            ),
            labels.reshape(-1),
            ignore_index=-100,
        )

    @staticmethod
    def compute_accuracy(
        logits: torch.Tensor,
        clean_ids: torch.Tensor,
        corrupted_positions: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
    ) -> float:
        """
        Computes token reconstruction accuracy.

        Accuracy is evaluated only on:
          - corrupted positions (if provided),
          - non-padding positions (if attention_mask is provided).
        """

        if logits.ndim != 3:
            raise ValueError(
                "logits must have shape "
                "[batch, sequence, vocabulary]"
            )

        if clean_ids.ndim != 2:
            raise ValueError(
                "clean_ids must have shape "
                "[batch, sequence]"
            )

        predictions = logits.argmax(dim=-1)

        valid_positions = torch.ones_like(
            clean_ids,
            dtype=torch.bool,
        )

        if corrupted_positions is not None:
            valid_positions &= corrupted_positions.bool()

        if attention_mask is not None:
            valid_positions &= attention_mask.bool()

        if not valid_positions.any():
            return 0.0

        correct = (
            predictions[valid_positions]
            == clean_ids[valid_positions]
        ).float()

        return correct.mean().item()

    def _validate_inputs(
        self,
        corrupted_ids: torch.Tensor,
        timesteps: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> None:
        if corrupted_ids.ndim != 2:
            raise ValueError(
                "corrupted_ids must have shape "
                "[batch, sequence]"
            )

        if timesteps.ndim != 1:
            raise ValueError(
                "timesteps must have shape [batch]"
            )

        if (
            corrupted_ids.size(0)
            != timesteps.size(0)
        ):
            raise ValueError(
                "Batch size of corrupted_ids and "
                "timesteps must match"
            )

        if attention_mask is not None:
            if attention_mask.shape != corrupted_ids.shape:
                raise ValueError(
                    "attention_mask must have the same "
                    "shape as corrupted_ids"
                )

        if torch.any(timesteps < 0):
            raise ValueError(
                "Timesteps cannot be negative"
            )

        if torch.any(
            timesteps >= self.num_diffusion_steps
        ):
            raise ValueError(
                f"Timesteps must be smaller than "
                f"{self.num_diffusion_steps}"
            )


In [ ]:
def print_reconstructed(
    tokenizer,
    clean_ids,
    corrupted_ids,
    logits,
    corrupted_positions,
):
    predicted_ids = logits.argmax(dim=-1)

    reconstructed_ids = corrupted_ids.clone()

    reconstructed_ids[corrupted_positions] = predicted_ids[
        corrupted_positions
    ]

    for i in range(clean_ids.size(0)):
      print("Original:     ", tokenizer.decode(clean_ids[i]).split('<|pad|>')[0])
      print("Corrupted:    ", tokenizer.decode(corrupted_ids[i]).split('<|pad|>')[0])
      print("Reconstructed:", tokenizer.decode(reconstructed_ids[i]).split('<|pad|>')[0])
      print('-----------')

## Test of the implementation

In [ ]:
MODEL_NAME = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.add_special_tokens({
    "pad_token": "<|pad|>",
    "mask_token": "<|mask|>",
})

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

2

In [ ]:
model = GPT2DiffusionTransformer(
    model_name=MODEL_NAME,
    num_diffusion_steps=1000,
    vocabulary_size=len(tokenizer)
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
batch = tokenizer(
    [
        "The cat sat on the mat.",
        "A dog is running outside.",
    ],
    padding=True,
    return_tensors="pt",
)

clean_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

corrupted_ids = clean_ids.clone()
corrupted_positions = torch.zeros_like(
    clean_ids,
    dtype=torch.bool,
)

# Nie maskujemy paddingu ani specjalnych tokenów.
corrupted_positions[0, 2] = True
corrupted_positions[0, 4] = True
corrupted_positions[1, 3] = True

corrupted_ids[corrupted_positions] = tokenizer.mask_token_id

timesteps = torch.tensor([100, 700])

logits = model(
    corrupted_ids=corrupted_ids,
    timesteps=timesteps,
    attention_mask=attention_mask
)

print("Input shape:", corrupted_ids.shape)
print("Logits shape:", logits.shape)
print("Loss:", GPT2DiffusionTransformer.compute_loss(
    logits=logits,
    clean_ids=clean_ids,
    corrupted_positions=corrupted_positions,
    attention_mask=attention_mask,
))

print_reconstructed(
    tokenizer = tokenizer,
    clean_ids = clean_ids,
    corrupted_ids = corrupted_ids,
    logits = logits,
    corrupted_positions = corrupted_positions
)


Input shape: torch.Size([2, 7])
Logits shape: torch.Size([2, 7, 50259])
Loss: tensor(8.6845, grad_fn=<NllLossBackward0>)
Original:      The cat sat on the mat.
Corrupted:     The cat<|mask|> on<|mask|> mat.
Reconstructed: The cat, on, mat.
-----------
Original:      A dog is running outside.
Corrupted:     A dog is<|mask|> outside.
Reconstructed: A dog is- outside.
-----------


In [ ]:
ids = model.denoise(
    corrupted_ids=corrupted_ids,
    attention_mask=attention_mask,
    corrupted_positions=corrupted_positions,
    num_iterations=10,
    temperature=1.0,
)

print(tokenizer.decode(ids[0]).split('<|pad|>')[0])

The cat or on, mat.


## Training loop
### Interfejs korupcji

In [ ]:
@dataclass
class CorruptionOutput:
    corrupted_ids: torch.Tensor
    corrupted_positions: torch.Tensor

class CorruptionMethod(ABC):
    @abstractmethod
    def __call__(
        self,
        clean_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> CorruptionOutput:
        pass

### Szczegolny metody korupcji

In [ ]:
class MaskTokenCorruption(CorruptionMethod):
    def __init__(
        self,
        mask_token_id: int,
        num_diffusion_steps: int,
        minimum_probability: float = 0.01,
        maximum_probability: float = 0.95,
    ) -> None:
        self.mask_token_id = mask_token_id
        self.num_diffusion_steps = num_diffusion_steps
        self.minimum_probability = minimum_probability
        self.maximum_probability = maximum_probability

    def __str__(self, ) -> str:
        return 'mask'

    def __call__(
        self,
        clean_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> CorruptionOutput:
        corrupted_ids = clean_ids.clone()

        normalized_t = timesteps.float() / (
            self.num_diffusion_steps - 1
        )

        probabilities = (
            self.minimum_probability
            + normalized_t
            * (
                self.maximum_probability
                - self.minimum_probability
            )
        )

        # [batch] -> [batch, 1]
        probabilities = probabilities.unsqueeze(1)

        random_values = torch.rand(
            clean_ids.shape,
            device=clean_ids.device,
        )

        corrupted_positions = (
            random_values < probabilities
        )

        # Nie psujemy paddingu.
        corrupted_positions &= attention_mask.bool()

        if not corrupted_positions.any():
          valid_positions = attention_mask.bool().nonzero(as_tuple=False)
          if valid_positions.numel() == 0:
              # If no valid positions to corrupt, return the original clean_ids
              # and an empty corrupted_positions mask for this batch item.
              return CorruptionOutput(
                  corrupted_ids=clean_ids,
                  corrupted_positions=torch.zeros_like(clean_ids, dtype=torch.bool),
              )
          random_index = torch.randint(
              low=0,
              high=valid_positions.size(0),
              size=(1,),
              device=clean_ids.device,
          )
          batch_idx, token_idx = valid_positions[random_index].squeeze(0)
          corrupted_positions[batch_idx, token_idx] = True

        corrupted_ids[corrupted_positions] = self.mask_token_id

        return CorruptionOutput(
            corrupted_ids=corrupted_ids,
            corrupted_positions=corrupted_positions,
        )

In [ ]:
class RandomTokenCorruption(CorruptionMethod):
    def __init__(
        self,
        num_diffusion_steps: int,
        dictionary_size: int,
        minimum_probability: float = 0.01,
        maximum_probability: float = 0.95,
    ) -> None:
        self.num_diffusion_steps = num_diffusion_steps
        self.minimum_probability = minimum_probability
        self.maximum_probability = maximum_probability
        self.dictionary_size = dictionary_size

    def __str__(self, ) -> str:
        return 'random'

    def __call__(
        self,
        clean_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> CorruptionOutput:
        corrupted_ids = clean_ids.clone()

        normalized_t = timesteps.float() / (
            self.num_diffusion_steps - 1
        )

        probabilities = (
            self.minimum_probability
            + normalized_t
            * (
                self.maximum_probability
                - self.minimum_probability
            )
        )

        # [batch] -> [batch, 1]
        probabilities = probabilities.unsqueeze(1)

        random_values = torch.rand(
            clean_ids.shape,
            device=clean_ids.device,
        )

        corrupted_positions = (
            random_values < probabilities
        )

        # Nie psujemy paddingu.
        corrupted_positions &= attention_mask.bool()

        if not corrupted_positions.any():
          valid_positions = attention_mask.bool().nonzero(as_tuple=False)
          if valid_positions.numel() == 0:
              # If no valid positions to corrupt, return the original clean_ids
              # and an empty corrupted_positions mask for this batch item.
              return CorruptionOutput(
                  corrupted_ids=clean_ids,
                  corrupted_positions=torch.zeros_like(clean_ids, dtype=torch.bool),
              )
          random_index = torch.randint(
              low=0,
              high=valid_positions.size(0),
              size=(1,),
              device=clean_ids.device,
          )
          batch_idx, token_idx = valid_positions[random_index].squeeze(0)
          corrupted_positions[batch_idx, token_idx] = True

        number_of_corrupted_tokens = int(
          corrupted_positions.sum().item()
        )

        corrupted_ids[corrupted_positions] = torch.randint(
            low=0,
            high=self.dictionary_size,
            size=(number_of_corrupted_tokens,),
            dtype=clean_ids.dtype,
            device=clean_ids.device,
        )

        return CorruptionOutput(
            corrupted_ids=corrupted_ids,
            corrupted_positions=corrupted_positions,
        )

In [ ]:
class SimilarTokenCorruption(CorruptionMethod):
    def __init__(
        self,
        num_diffusion_steps: int,
        embedding_weight: torch.Tensor,
        number_of_neighbors: int = 20,
        minimum_probability: float = 0.01,
        maximum_probability: float = 0.30,
        chunk_size: int = 1024,
    ) -> None:
        self.num_diffusion_steps = num_diffusion_steps
        self.minimum_probability = minimum_probability
        self.maximum_probability = maximum_probability
        self.number_of_neighbors = number_of_neighbors

        self.nearest_token_ids = self._build_nearest_token_table(
            embedding_weight=embedding_weight,
            number_of_neighbors=number_of_neighbors,
            chunk_size=chunk_size,
        )

    def __str__(self, ) -> str:
        return 'similar'

    @staticmethod
    @torch.no_grad()
    def _build_nearest_token_table(
        embedding_weight: torch.Tensor,
        number_of_neighbors: int,
        chunk_size: int,
    ) -> torch.Tensor:
        """
        Buduje tabelę najbliższych tokenów według cosine similarity.

        Zwraca:
            Tensor o kształcie:
            [vocabulary_size, number_of_neighbors]
        """
        if embedding_weight.ndim != 2:
            raise ValueError(
                "embedding_weight must have shape "
                "[vocabulary_size, embedding_size]"
            )

        if number_of_neighbors <= 0:
            raise ValueError(
                "number_of_neighbors must be greater than zero"
            )

        if chunk_size <= 0:
            raise ValueError(
                "chunk_size must be greater than zero"
            )

        vocabulary_size = embedding_weight.size(0)

        if number_of_neighbors >= vocabulary_size:
            raise ValueError(
                "number_of_neighbors must be smaller "
                "than vocabulary size"
            )

        device = embedding_weight.device

        normalized_embeddings = F.normalize(
            embedding_weight.detach().float(),
            p=2,
            dim=-1,
        )

        nearest_chunks = []

        for start in (range(0, vocabulary_size, chunk_size)):
            end = min(
                start + chunk_size,
                vocabulary_size,
            )

            current_embeddings = normalized_embeddings[
                start:end
            ]

            similarities = (
                current_embeddings
                @ normalized_embeddings.T
            )

            local_row_indices = torch.arange(
                end - start,
                device=device,
            )

            global_token_indices = torch.arange(
                start,
                end,
                device=device,
            )

            # Token nie może być własnym sąsiadem.
            similarities[
                local_row_indices,
                global_token_indices,
            ] = -torch.inf

            nearest_ids = torch.topk(
                similarities,
                k=number_of_neighbors,
                dim=-1,
            ).indices

            nearest_chunks.append(
                nearest_ids.cpu()
            )

        return torch.cat(
            nearest_chunks,
            dim=0,
        )

    def __call__(
        self,
        clean_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> CorruptionOutput:
        corrupted_ids = clean_ids.clone()

        normalized_t = timesteps.float() / (
            self.num_diffusion_steps - 1
        )

        probabilities = (
            self.minimum_probability
            + normalized_t
            * (
                self.maximum_probability
                - self.minimum_probability
            )
        )

        probabilities = probabilities.unsqueeze(1)

        random_values = torch.rand(
            clean_ids.shape,
            device=clean_ids.device,
        )

        corrupted_positions = (
            random_values < probabilities
        )

        # Nie psujemy paddingu.
        corrupted_positions &= attention_mask.bool()

        # Gwarantujemy przynajmniej jedną korupcję w batchu.
        if not corrupted_positions.any():
            valid_positions = attention_mask.bool().nonzero(
                as_tuple=False
            )

            if valid_positions.numel() == 0:
                # If no valid positions to corrupt, return the original clean_ids
                # and an empty corrupted_positions mask for this batch item.
                return CorruptionOutput(
                    corrupted_ids=clean_ids,
                    corrupted_positions=torch.zeros_like(clean_ids, dtype=torch.bool),
                )

            random_index = torch.randint(
                low=0,
                high=valid_positions.size(0),
                size=(1,),
                device=clean_ids.device,
            )

            batch_idx, token_idx = valid_positions[
                random_index
            ].squeeze(0)

            corrupted_positions[
                batch_idx,
                token_idx,
            ] = True

        original_token_ids = clean_ids[
            corrupted_positions
        ]

        nearest_token_ids = self.nearest_token_ids.to(
            clean_ids.device
        )

        candidate_neighbors = nearest_token_ids[
            original_token_ids
        ]

        number_of_corrupted_tokens = (
            original_token_ids.numel()
        )

        random_neighbor_indices = torch.randint(
            low=0,
            high=self.number_of_neighbors,
            size=(number_of_corrupted_tokens,),
            device=clean_ids.device,
        )

        row_indices = torch.arange(
            number_of_corrupted_tokens,
            device=clean_ids.device,
        )

        replacement_token_ids = candidate_neighbors[
            row_indices,
            random_neighbor_indices,
        ]

        corrupted_ids[
            corrupted_positions
        ] = replacement_token_ids

        return CorruptionOutput(
            corrupted_ids=corrupted_ids,
            corrupted_positions=corrupted_positions,
        )

In [ ]:
class MixedTokenCorruption(CorruptionMethod):
    def __init__(
        self,
        corruption_methods: list[CorruptionMethod],
        iterations_intervals: list[int],
    ) -> None:
        self.corruption_methods = corruption_methods
        self.iterations_intervals = iterations_intervals
        self.counter = 0

    def __str__(self) -> str:
        total_iterations_sum = 0
        for i in range(len(self.iterations_intervals)):
            total_iterations_sum += self.iterations_intervals[i]
            if self.counter < total_iterations_sum:
                return str(self.corruption_methods[i])
        # If the counter exceeds all intervals, return the string representation of the last method
        return str(self.corruption_methods[-1])

    def __call__(
        self,
        clean_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> CorruptionOutput:
        corruption_method = self.corruption_methods[-1]

        for i in range(len(self.iterations_intervals)):
            if self.counter < sum(
                self.iterations_intervals[:i + 1]
            ):
                corruption_method = self.corruption_methods[i]
                break

        self.counter += 1

        return corruption_method(
            clean_ids=clean_ids,
            attention_mask=attention_mask,
            timesteps=timesteps,
        )

### Collator

In [ ]:
class DiffusionDataCollator:
    def __init__(
        self,
        tokenizer,
        corruption_method: CorruptionMethod,
        num_diffusion_steps: int,
        max_length: int = 128,
    ) -> None:
        self.tokenizer = tokenizer
        self.corruption_method = corruption_method
        self.num_diffusion_steps = num_diffusion_steps
        self.max_length = max_length

    def __call__(self, examples: list[dict]) -> dict[str, torch.Tensor]:
        texts = [example["text"] for example in examples]

        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        clean_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        # Filter out samples where attention_mask is all zeros (effectively empty sequences)
        valid_sample_indices = (attention_mask.sum(dim=1) > 0).nonzero(as_tuple=True)[0]

        if len(valid_sample_indices) == 0:
            # Return an empty batch that won't cause issues downstream
            return {
                "clean_ids": torch.empty(0, 0, dtype=clean_ids.dtype),
                "corrupted_ids": torch.empty(0, 0, dtype=clean_ids.dtype),
                "corrupted_positions": torch.empty(0, 0, dtype=torch.bool),
                "attention_mask": torch.empty(0, 0, dtype=attention_mask.dtype),
                "timesteps": torch.empty(0, dtype=torch.long),
            }

        clean_ids = clean_ids[valid_sample_indices]
        attention_mask = attention_mask[valid_sample_indices]

        batch_size = clean_ids.size(0)

        timesteps = torch.randint(
            low=0,
            high=self.num_diffusion_steps,
            size=(batch_size,),
        )

        corruption = self.corruption_method(
            clean_ids=clean_ids,
            attention_mask=attention_mask,
            timesteps=timesteps,
        )

        return {
            "clean_ids": clean_ids,
            "corrupted_ids": corruption.corrupted_ids,
            "corrupted_positions": corruption.corrupted_positions,
            "attention_mask": attention_mask,
            "timesteps": timesteps,
        }

### Dataset

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts: list[str]) -> None:
        self.texts = texts

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict[str, str]:
        return {
            "text": self.texts[index],
        }

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

In [ ]:
ds

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

### Dataset loader

In [ ]:
dataset = TextDataset([
    "The cat sat on the mat.",
    "A dog is running outside.",
    "Transformers can reconstruct corrupted sequences.",
    "Diffusion models learn to reverse a corruption process.",
])

dataset_test = ds['test']
dataset_train = ds['train']

corruption_method_1 = SimilarTokenCorruption(
    embedding_weight=model.transformer.wte.weight,
    num_diffusion_steps=100,
    number_of_neighbors=20,
    minimum_probability=0.01,
    maximum_probability=0.30,
)

corruption_method_2 = RandomTokenCorruption(
    dictionary_size=len(tokenizer),
    num_diffusion_steps=100,
    minimum_probability=0.01,
    maximum_probability=0.95,
)

corruption_method_3 = MaskTokenCorruption(
    mask_token_id=tokenizer.mask_token_id,
    num_diffusion_steps=100,
    minimum_probability=0.01,
    maximum_probability=0.95,
)

iterations_intervals = [5, 2, 2]

corruption_method = MixedTokenCorruption(
    corruption_methods=[
        corruption_method_1,
        corruption_method_2,
        corruption_method_3,
    ],
    iterations_intervals=iterations_intervals,
)

collator = DiffusionDataCollator(
    tokenizer=tokenizer,
    corruption_method=corruption_method,
    num_diffusion_steps=100,
    max_length=64,
)

In [ ]:
train_loader = DataLoader(
    dataset_train,
    batch_size=95,
    shuffle=True,
    collate_fn=collator,
)

test_loader = DataLoader(
    dataset_test,
    batch_size=95,
    shuffle=True,
    collate_fn=collator,
)

### Training loop

In [ ]:
@dataclass
class TrainingOutput:
  train_loss: list[float]
  test_loss: list[float]
  train_acc: list[float]
  test_acc: list[float]
  iterations_intervals: dict

def train(device, model, optimizer, num_epochs, train_loader, test_loader=None) -> TrainingOutput:
    model = model.to(device)
    model.train()

    train_loss = []
    test_loss = []
    train_acc = []
    test_acc = []

    for epoch in range(num_epochs):
        total_loss_train = 0.0
        total_loss_test = 0.0

        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs}",
            leave=True,
            dynamic_ncols=True,
        )

        for batch in progress_bar:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            current_batch_attention_mask = batch["attention_mask"]
            valid_samples_in_batch = (
                current_batch_attention_mask.sum(dim=1) > 0
            ).nonzero(as_tuple=True)[0]

            if len(valid_samples_in_batch) == 0:
                continue

            for key in batch:
                batch[key] = batch[key][valid_samples_in_batch]

            optimizer.zero_grad(set_to_none=True)

            logits = model(
                corrupted_ids=batch["corrupted_ids"],
                timesteps=batch["timesteps"],
                attention_mask=batch["attention_mask"],
            )

            loss = model.compute_loss(
                logits=logits,
                clean_ids=batch["clean_ids"],
                corrupted_positions=batch["corrupted_positions"],
                attention_mask=batch["attention_mask"],
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            accuracy = model.compute_accuracy(
                logits=logits,
                clean_ids=batch["clean_ids"],
                corrupted_positions=batch["corrupted_positions"],
                attention_mask=batch["attention_mask"],
            )

            train_loss.append(loss.item())
            train_acc.append(accuracy)

            total_loss_train += loss.item()

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{accuracy:.4f}",
                corruption_method=f"{str(train_loader.collate_fn.corruption_method)}"
            )

        if test_loader:
            for batch in tqdm(
                test_loader,
                desc=f"Testing {epoch + 1}/{num_epochs}",
                leave=False,
                dynamic_ncols=True,
            ):
                batch = {
                    key: value.to(device)
                    for key, value in batch.items()
                }

                current_batch_attention_mask = batch["attention_mask"]
                valid_samples_in_batch = (
                    current_batch_attention_mask.sum(dim=1) > 0
                ).nonzero(as_tuple=True)[0]

                if len(valid_samples_in_batch) == 0:
                    continue

                for key in batch:
                    batch[key] = batch[key][valid_samples_in_batch]

                with torch.no_grad():
                    logits = model(
                        corrupted_ids=batch["corrupted_ids"],
                        timesteps=batch["timesteps"],
                        attention_mask=batch["attention_mask"],
                    )

                    loss = model.compute_loss(
                        logits=logits,
                        clean_ids=batch["clean_ids"],
                        corrupted_positions=batch["corrupted_positions"],
                        attention_mask=batch["attention_mask"],
                    )

                    accuracy = model.compute_accuracy(
                        logits=logits,
                        clean_ids=batch["clean_ids"],
                        corrupted_positions=batch["corrupted_positions"],
                        attention_mask=batch["attention_mask"],
                    )

                    test_loss.append(loss.item())
                    test_acc.append(accuracy)

                total_loss_test += loss.item()

        average_loss_train = total_loss_train / len(train_loader)
        average_loss_test = total_loss_test / len(test_loader) if test_loader else 0

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"train_loss={average_loss_train:.4f} | "
            f"test_loss={average_loss_test:.4f} | "
            f"train_acc={sum(train_acc) / len(train_acc):.4f} | "
            f"test_acc={sum(test_acc) / len(test_acc):.4f} | "
            f"corruption_method={str(train_loader.collate_fn.corruption_method)}"
        )

    return TrainingOutput(
        train_loss=train_loss,
        test_loss=test_loss,
        train_acc=train_acc,
        test_acc=test_acc,
        iterations_intervals=train_loader.collate_fn.corruption_method.iterations_intervals,
    )

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
)

num_epochs = sum(iterations_intervals)

training_output = train(device, model, optimizer, num_epochs, train_loader, test_loader)

Epoch 1/9:   3%|▎         | 12/387 [00:12<06:26,  1.03s/it, acc=0.0486, corruption_method=mask, loss=7.8538]


KeyboardInterrupt: 

In [ ]:
model.to('cpu')
batch = tokenizer(
    [
        "The cat sat on the mat.",
        "A dog is running outside.",
    ],
    padding=True,
    return_tensors="pt",
)

clean_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

corrupted_ids = clean_ids.clone()
corrupted_positions = torch.zeros_like(
    clean_ids,
    dtype=torch.bool,
)

# Nie maskujemy paddingu ani specjalnych tokenów.
corrupted_positions[0, 2] = True
corrupted_positions[0, 4] = True
corrupted_positions[1, 3] = True

corrupted_ids[corrupted_positions] = tokenizer.mask_token_id

timesteps = torch.tensor([100, 700])

logits = model(
    corrupted_ids=corrupted_ids,
    timesteps=timesteps,
    attention_mask=attention_mask
)

print("Input shape:", corrupted_ids.shape)
print("Logits shape:", logits.shape)
print("Loss:", GPT2DiffusionTransformer.compute_loss(
    logits=logits,
    clean_ids=clean_ids,
    corrupted_positions=corrupted_positions,
    attention_mask=attention_mask,
))

print_reconstructed(
    tokenizer = tokenizer,
    clean_ids = clean_ids,
    corrupted_ids = corrupted_ids,
    logits = logits,
    corrupted_positions = corrupted_positions
)


In [ ]:
ids = model.denoise(
    corrupted_ids=corrupted_ids,
    attention_mask=attention_mask,
    corrupted_positions=corrupted_positions,
    num_iterations=10,
    temperature=1.0,
)

print(tokenizer.decode(ids[0]).split('<|pad|>')[0])